In [1]:
# This notebook is used to generate the mixed datasets
import pandas as pd
import ndjson
import os

In [2]:
# - reward-bench
splits = {'raw': 'data/raw-00000-of-00001.parquet', 'filtered': 'data/filtered-00000-of-00001.parquet'}
data_reward_bench = pd.read_parquet("hf://datasets/allenai/reward-bench/" + splits["filtered"])
selected_data_reward_bench = data_reward_bench.loc[data_reward_bench['subset'].str.startswith("xstest-") | data_reward_bench['subset'].str.startswith("refusals-") | data_reward_bench['subset'].str.startswith("donotanswer") | data_reward_bench['subset'].str.startswith("hep-")
]
selected_data_reward_bench["new_id"] = selected_data_reward_bench["subset"] + "/" + selected_data_reward_bench["id"].astype(str)
selected_data_reward_bench = selected_data_reward_bench[['new_id', 'prompt']].rename(columns={'new_id': 'id'})

/Users/dalton386/Desktop/LLMRouting/Codes/adaptive_llm_routing/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/p3/bpkqcww548192ylbj0g_z8th0000gn/T/ipykernel_37650/1427453898.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_data_reward_bench["new_id"] = selected_data_reward_bench["subset"] + "/" + selected_data_reward_bench["id"].astype(str)


In [3]:
# - CodeUltraFeedback
data_code_ultra_feedback = pd.read_parquet("hf://datasets/coseal/CodeUltraFeedback/data/train-00000-of-00001.parquet")
selected_data_code_ultra_feedback = data_code_ultra_feedback[['instruction']].reset_index()
selected_data_code_ultra_feedback["id"] = 'code_ultra_feedback/' + selected_data_code_ultra_feedback["index"].astype(str)
selected_data_code_ultra_feedback = selected_data_code_ultra_feedback[['id', 'instruction']].rename(columns={'instruction': 'prompt'})
selected_data_code_ultra_feedback = selected_data_code_ultra_feedback.sample(n=1016, random_state=1)

In [4]:
# - mix-instruct
splits = {'train': 'train_data_prepared.jsonl', 'validation': 'val_data_prepared.jsonl', 'test': 'test_data_prepared.jsonl'}
data_mix_instruct = pd.read_json("hf://datasets/llm-blender/mix-instruct/" + splits["train"], lines=True)
selected_data_mix_instruct = data_mix_instruct[['id', 'instruction', 'input']]
selected_data_mix_instruct['prompt'] = selected_data_mix_instruct['instruction'] + ' ' + selected_data_mix_instruct['input']
selected_data_mix_instruct = selected_data_mix_instruct[['id', 'prompt']]
selected_data_mix_instruct = selected_data_mix_instruct.sample(n=6000, random_state=1)

/var/folders/p3/bpkqcww548192ylbj0g_z8th0000gn/T/ipykernel_37650/1389323287.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_data_mix_instruct['prompt'] = selected_data_mix_instruct['instruction'] + ' ' + selected_data_mix_instruct['input']


In [5]:
# - BeaverTails
splits = {'330k_train': 'round0/330k/train.jsonl.xz', '330k_test': 'round0/330k/test.jsonl.xz', '30k_train': 'round0/30k/train.jsonl.gz', '30k_test': 'round0/30k/test.jsonl.gz'}
data_beaver_tails = pd.read_json("hf://datasets/PKU-Alignment/BeaverTails/" + splits["30k_train"], lines=True)

def is_prompt_harmful(category):
    for k, v in category.items():
        if v:
            return True
    return False

# select only the prompts that are harmful
selected_data_beaver_tails = data_beaver_tails[data_beaver_tails['category'].apply(is_prompt_harmful)]
selected_data_beaver_tails = selected_data_beaver_tails[['prompt']].drop_duplicates().reset_index()
selected_data_beaver_tails["id"] = 'beaver_tails/' + selected_data_beaver_tails["index"].astype(str)
selected_data_beaver_tails = selected_data_beaver_tails[['id', 'prompt']]
selected_data_beaver_tails = selected_data_beaver_tails.sample(n=1260, random_state=1)

In [7]:
# save the selected data as a JSONL file
selected_data = pd.concat([selected_data_reward_bench, selected_data_code_ultra_feedback, selected_data_beaver_tails, selected_data_mix_instruct])
selected_data_dict = selected_data.to_dict(orient='records')

file_path = '../data/mixed_dataset.jsonl'
os.makedirs(os.path.dirname(file_path), exist_ok=True)
with open(file_path, 'w', encoding='utf8') as f:
    ndjson.dump(selected_data_dict, f, ensure_ascii=False)